# Strategic Customer Retention via Deep Learning

**StreamNet Churn Prediction | Deep Learning Ensemble | Multi-Class Classification**

---

## Overview

This project builds a deep learning ensemble to predict which retention strategy to apply to individual streaming service subscribers. Given a labeled dataset of customer profiles and behavioral signals, the model classifies each customer into one of four strategic actions: Monitor, Re-Engage, Discount, or Terminate.

The final solution uses a majority-vote ensemble of three independently trained neural networks, achieving over 90% validation accuracy on a held-out set, and generates predictions for 10,000 unlabeled customers.

## Approach

1. Feature engineering on demographic, behavioral, and device signals
2. Hyperparameter search across architectures (layer depth, dropout rates, activation functions, batch sizes)
3. Ensemble construction using the top 3 configurations by validation accuracy
4. Majority-vote prediction with fallback to the best single model if the ensemble underperforms

## Stack

TensorFlow / Keras, scikit-learn, NumPy, pandas, SciPy

In [ ]:
import numpy as np
import pandas as pd
import random
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import Dense, Dropout, Input, BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from scipy import stats as scipy_stats

SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('Libraries loaded.')


## Data Loading

The dataset contains labeled and unlabeled customer records. Rows with a known `strategic_action` label form the training set. The 10,000 rows without a label are the prediction targets.

In [ ]:
df = pd.read_csv('Project 01 - Data.csv')
print(f'Total rows: {len(df):,}')
print(df['strategic_action'].value_counts(dropna=False))
df.head()


In [ ]:
df_train = df[df['strategic_action'].notna()].copy()
df_pred  = df[df['strategic_action'].isna()].copy()
pred_ids = df_pred['id'].values
print(f'Training rows:   {len(df_train):,}')
print(f'Prediction rows: {len(df_pred):,}')
print(f'Prediction IDs:  {pred_ids.min()} to {pred_ids.max()}')


## Feature Engineering

Categorical variables (`gender`, `region`) are one-hot encoded. Interaction features are constructed to capture non-linear relationships between engagement, tenure, income, age, and device count. These interaction terms consistently improved validation accuracy during development.

In [ ]:
df_train = pd.get_dummies(df_train, columns=['gender','region'], drop_first=False)
df_pred  = pd.get_dummies(df_pred,  columns=['gender','region'], drop_first=False)

for d in [df_train, df_pred]:
    d['eng_tenure'] = d['engagement_score'] * d['tenure_months']
    d['inc_device'] = d['estimated_income'] / (d['device_count'] + 1)
    d['age_eng']    = d['age'] * d['engagement_score']
    d['inc_eng']    = d['estimated_income'] * d['engagement_score']
    d['age_device'] = d['age'] * d['device_count']
    d['eng_sq']     = d['engagement_score'] ** 2

feature_cols = [c for c in df_train.columns if c not in ['id','strategic_action']]
X      = df_train[feature_cols].values
y      = df_train['strategic_action'].values.astype(int)
X_pred = df_pred[feature_cols].values
print(f'Features: {len(feature_cols)}, X shape: {X.shape}')


## Preprocessing

All features are normalized using z-score scaling. The scaler is fit on training data only and applied to the prediction set to prevent data leakage.

In [ ]:
scaler = StandardScaler()
X      = scaler.fit_transform(X)
X_pred = scaler.transform(X_pred)
print('Normalized. Mean:', round(X.mean(), 4), 'Std:', round(X.std(), 4))


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y)
n_features = X_train.shape[1]
n_classes  = 4
print(f'Train: {X_train.shape}, Val: {X_val.shape}')


## Model Architecture and Training

Three configurations were selected from hyperparameter search based on validation accuracy. Each uses a 5-layer feed-forward network with batch normalization and dropout for regularization. The top configurations by validation accuracy were:

- `batch_512`: ELU activations, batch size 512 (90.41%)
- `high_dropout`: ReLU activations, aggressive dropout (90.34%)
- `baseline`: ELU activations, batch size 256 (90.30%)

All models use early stopping on validation accuracy (patience=25) and learning rate reduction on plateau.

In [ ]:
top_configs = [
    {'name': 'batch_512',
     'layers': [1024, 512, 256, 128, 64],
     'dropouts': [0.30, 0.25, 0.20, 0.15, 0.10],
     'activation': 'elu', 'lr': 0.001, 'batch_size': 512},
    {'name': 'high_dropout',
     'layers': [1024, 512, 256, 128, 64],
     'dropouts': [0.50, 0.40, 0.30, 0.20, 0.10],
     'activation': 'relu', 'lr': 0.001, 'batch_size': 256},
    {'name': 'baseline',
     'layers': [1024, 512, 256, 128, 64],
     'dropouts': [0.30, 0.25, 0.20, 0.15, 0.10],
     'activation': 'elu', 'lr': 0.001, 'batch_size': 256},
]

def build_and_train(cfg, seed_val):
    np.random.seed(seed_val)
    random.seed(seed_val)
    tf.random.set_seed(seed_val)

    inp_ = Input(shape=(n_features,))
    x_ = inp_
    for i, units in enumerate(cfg['layers']):
        x_ = Dense(units, activation=cfg['activation'])(x_)
        x_ = BatchNormalization()(x_)
        dr = cfg['dropouts'][i]
        x_ = Dropout(dr)(x_)
    out_ = Dense(n_classes, activation='softmax')(x_)
    m = Model(inputs=inp_, outputs=out_)
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg['lr']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    cbs = [
        EarlyStopping(monitor='val_accuracy', patience=25,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                          patience=8, min_lr=1e-6, verbose=0)
    ]
    h = m.fit(X_train, y_train,
              validation_data=(X_val, y_val),
              epochs=500, batch_size=cfg['batch_size'],
              callbacks=cbs, verbose=0)
    return m, max(h.history['val_accuracy'])

models = []
accs   = []

for i, cfg in enumerate(top_configs):
    print(f'Training model {i+1} of 3: {cfg["name"]}...')
    m, acc = build_and_train(cfg, seed_val=42)
    models.append(m)
    accs.append(acc)
    print(f'  Val accuracy: {acc*100:.2f}%')

best_single_acc = max(accs)
best_model      = models[int(np.argmax(accs))]
print(f'\nBest single model: {best_single_acc*100:.2f}%')


## Ensemble: Majority Vote

Predictions from all three models are combined via majority vote. The ensemble is used if it matches or exceeds the best individual model's validation accuracy.

In [ ]:
val_votes = np.array([
    np.argmax(m.predict(X_val, verbose=0), axis=1) for m in models
])
ensemble_val, _ = scipy_stats.mode(val_votes, axis=0)
ensemble_acc    = accuracy_score(y_val, ensemble_val.flatten())

print(f'Best single model : {best_single_acc*100:.2f}%')
print(f'Ensemble accuracy : {ensemble_acc*100:.2f}%')

use_ensemble = ensemble_acc >= best_single_acc
best_val_acc = ensemble_acc if use_ensemble else best_single_acc
print(f'\nUsing: {"ensemble" if use_ensemble else "best single model"}')
print(f'Final accuracy: {best_val_acc*100:.2f}%')


## Evaluation

Per-class precision, recall, and F1 on the validation set.

In [ ]:
if use_ensemble:
    report_preds = ensemble_val.flatten()
else:
    report_preds = np.argmax(best_model.predict(X_val, verbose=0), axis=1)

print(classification_report(y_val, report_preds,
      target_names=['0-Monitor','1-ReEngage','2-Discount','3-Terminate']))


## Generating Predictions

The final model (ensemble or best single) generates retention strategy predictions for the 10,000 unlabeled customers.

In [ ]:
if use_ensemble:
    pred_votes = np.array([
        np.argmax(m.predict(X_pred, verbose=0), axis=1) for m in models
    ])
    preds, _ = scipy_stats.mode(pred_votes, axis=0)
    preds    = preds.flatten().astype(int)
    print('Method: ensemble majority vote')
else:
    preds = np.argmax(best_model.predict(X_pred, verbose=0), axis=1)
    print('Method: best single model')

print(f'Total predictions: {len(preds):,}')
print('Prediction distribution:')
for i, count in enumerate(np.bincount(preds)):
    print(f'  Class {i}: {count:,} ({count/len(preds)*100:.1f}%)')
assert len(preds) == 10000, f'Expected 10000, got {len(preds)}'
print('10,000 predictions confirmed.')


## Export and Validation

In [ ]:
df_out = pd.DataFrame({'id': pred_ids, 'strategic_action': preds})
output_filename = 'Group 11.csv'
df_out.to_csv(output_filename, columns=['id','strategic_action'],
              header=False, index=False)
print(f'Saved: {output_filename}')
print(f'Rows: {len(df_out):,}')
print(df_out.head(10).to_string(index=False))


In [ ]:
df_check      = pd.read_csv(output_filename, header=None, names=['id','strategic_action'])
expected_ids  = set(range(100001, 110001))
submitted_ids = set(df_check['id'].astype(int))
first_line    = open(output_filename).readline().lower()

print('=== OUTPUT VALIDATION ===')
print(f'Row count     : {len(df_check):,} - {"PASS" if len(df_check)==10000 else "FAIL"}')
print(f'No headers    : {"PASS" if "id" not in first_line and "strategic" not in first_line else "FAIL"}')
print(f'Categories    : {sorted(df_check["strategic_action"].unique())}')
print(f'Missing IDs   : {len(expected_ids-submitted_ids)} - {"PASS" if not (expected_ids-submitted_ids) else "FAIL"}')
print(f'Extra IDs     : {len(submitted_ids-expected_ids)} - {"PASS" if not (submitted_ids-expected_ids) else "FAIL"}')
print(f'Duplicate IDs : {df_check["id"].duplicated().sum()} - {"PASS" if not df_check["id"].duplicated().any() else "FAIL"}')
print(f'\nFinal accuracy : {best_val_acc*100:.2f}%')
